# Parameter-shift gradients
**Quantum Computing · Week 7 (Variational quantum circuits) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/qc/w7d3.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Training needs slopes. On a quantum computer we can't backpropagate, but the **parameter-shift rule**
gives exact slopes from two extra runs of the same circuit:
$\frac{\partial C}{\partial\theta} = \frac12\big[C(\theta + \frac\pi2) - C(\theta - \frac\pi2)\big]$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

theta = Parameter("θ")
one = QuantumCircuit(1)
one.rx(0.9, 0)
one.ry(theta, 0)
Z = SparsePauliOp("Z")
estimator = StatevectorEstimator()
C = lambda t: estimator.run([(one, Z, [t])]).result()[0].data.evs.item()

t0 = 0.6
shift = (C(t0 + np.pi / 2) - C(t0 - np.pi / 2)) / 2
tiny = (C(t0 + 1e-6) - C(t0 - 1e-6)) / 2e-6             # a very small step, fine without shot noise
print(f"parameter shift: {shift:.8f}")
print(f"tiny finite step: {tiny:.8f}")

They agree to many digits. The shift rule isn't an approximation: because the cost is a sine wave in
$\theta$, half the difference a quarter-turn either side is exactly the slope.

## Every knob at once
For several knobs, shift one at a time: two evaluations per knob. Yesterday's two-knob circuit:

In [ ]:
H = SparsePauliOp.from_list([("ZZ", 1.0), ("IX", 0.5), ("XI", 0.5)])
t1, t2 = Parameter("θ1"), Parameter("θ2")
two = QuantumCircuit(2)
two.ry(t1, 0)
two.ry(t2, 1)
two.cx(0, 1)
energy = lambda th: estimator.run([(two, H, th)]).result()[0].data.evs.item()

def parameter_shift_gradient(f, theta):
    grad = np.zeros(len(theta))
    for k in range(len(theta)):
        shift = np.zeros(len(theta))
        shift[k] = np.pi / 2
        grad[k] = (f(theta + shift) - f(theta - shift)) / 2
    return grad

theta0 = np.array([0.3, 0.3])
print("gradient:", parameter_shift_gradient(energy, theta0).round(6), " (2 knobs: 4 circuit evaluations)")

A model with $P$ knobs needs $2P$ evaluations per gradient, each with many shots. Backpropagation on a
classical network gets all its gradients for about the price of one extra pass (ML week 5). That
difference is one of the practical costs of quantum machine learning.

## Shot noise: why the big shift wins
With shots, every evaluation is noisy. A finite difference divides that noise by a small $2h$, which
blows it up; the parameter shift divides by 2.

In [ ]:
shots = 1000
noisy = StatevectorEstimator(default_precision=1 / np.sqrt(shots), seed=np.random.default_rng(3))   # a generator: fresh noise every call
Cn = lambda t: noisy.run([(one, Z, [t])]).result()[0].data.evs.item()
truth = shift
ps = [(Cn(t0 + np.pi / 2) - Cn(t0 - np.pi / 2)) / 2 for _ in range(300)]
fd = {h: [(Cn(t0 + h) - Cn(t0 - h)) / (2 * h) for _ in range(300)] for h in [0.01, 0.1]}
print(f"true slope {truth:.3f}")
print(f"parameter shift:        mean {np.mean(ps):+.3f}, spread {np.std(ps):.3f}")
for h, values in fd.items():
    print(f"finite difference h={h}: mean {np.mean(values):+.3f}, spread {np.std(values):.3f}")

## Your turn: train with parameter shift
Run gradient descent on the two-knob circuit from $(0.3, 0.3)$: 40 steps of
$\boldsymbol\theta \leftarrow \boldsymbol\theta - 0.2\,\nabla E$, with gradients from `parameter_shift_gradient`.
Plot the energy per step, and count how many circuit evaluations the training used.

In [ ]:
theta = np.array([0.3, 0.3])
energies = [energy(theta)]
for step in range(40):
    pass          # TODO: theta = theta - 0.2 * parameter_shift_gradient(energy, theta); append the new energy
print(f"final energy {energies[-1]:.4f} (ground state {-np.sqrt(2):.4f})")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
theta = np.array([0.3, 0.3])
energies = [energy(theta)]
for step in range(40):
    theta = theta - 0.2 * parameter_shift_gradient(energy, theta)
    energies.append(energy(theta))
print(f"final energy {energies[-1]:.4f} (ground state {-np.sqrt(2):.4f}) after 40 steps")
print(f"circuit evaluations for the gradients: 40 steps x 2 knobs x 2 shifts = {40 * 2 * 2}")
plt.plot(energies, "o-", ms=4, color="#7b5cfa")
plt.axhline(-np.sqrt(2), color="#17a98c", ls="--", label="ground state")
plt.xlabel("step")
plt.ylabel("energy")
plt.legend()
plt.show()